# Story generator with an LSTM

Run top to bottom. Two things were changed after the session ran out of RAM:

1. The answers are kept as plain word numbers instead of one-hot rows
   (8.66 GB -> 0.69 MB). This is what crashed the session.
2. The per-operation device logging and the full-list prints were removed.
   They wrote millions of lines into the notebook while training.

In [29]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, LSTM, Bidirectional, Embedding
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [30]:
# Was: tf.debugging.set_log_device_placement(True)
# That printed one line for every operation TensorFlow ran. Across a full
# training run it produced millions of lines, which fills the notebook and
# the browser tab's memory on its own. One check is enough.
print("GPU:", tf.config.list_physical_devices('GPU'))

GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [31]:
path = '/content/stories.txt' if os.path.exists('/content/stories.txt') else 'stories.txt'
text = open(path, encoding='utf-8').read().lower()

print(f"{len(text.split()):,} words, {len(text.splitlines()):,} lines")

202,651 words, 40,000 lines


In [32]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts([text])
total_words = len(tokenizer.word_index) + 1

print("vocabulary size:", total_words)

vocabulary size: 12633


In [33]:
# Was: print(tokenizer.word_index) -- that dumps all ~12,600 words into the
# notebook. The first 20 are enough to confirm it worked.
print(list(tokenizer.word_index.items())[:20])

[('the', 1), ('and', 2), ('to', 3), ('i', 4), ('of', 5), ('you', 6), ('my', 7), ('a', 8), ('that', 9), ('in', 10), ('is', 11), ('not', 12), ('for', 13), ('with', 14), ('me', 15), ('it', 16), ('be', 17), ('your', 18), ('his', 19), ('but', 20)]


In [34]:
# Build the practice questions: every line gives "first i words -> word i+1".
# WINDOW caps how far back the model looks, so one long line cannot blow up
# the size of every padded row.
WINDOW = 16

input_sequence = []
for line in text.splitlines():
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        input_sequence.append(token_list[max(0, i + 1 - WINDOW):i + 1])

print(f"{len(input_sequence):,} training sequences")

171,312 training sequences


In [35]:
# Was: input_sequence -- printing the whole list is ~170,000 rows of output.
input_sequence[:5]

[[88, 269],
 [139, 35],
 [139, 35, 969],
 [139, 35, 969, 143],
 [139, 35, 969, 143, 668]]

In [36]:
max_length = max(len(x) for x in input_sequence)
input_pad = pad_sequences(input_sequence, maxlen=max_length, padding='pre', dtype='int32')

print("padded shape:", input_pad.shape, f"({input_pad.nbytes / 1e6:.1f} MB)")

padded shape: (171312, 16) (11.0 MB)


In [37]:
predictors, label = input_pad[:, :-1], input_pad[:, -1]

# THIS IS THE FIX FOR THE CRASH.
#
# The original line here was:
#     label = tf.keras.utils.to_categorical(label, num_classes=total_words)
#
# That turns every answer into a row of 12,633 numbers with a single 1 in it:
#     171,312 answers x 12,633 words x 4 bytes = 8.66 GB
# Colab has ~12 GB of RAM, so the session died before training started.
#
# Instead we keep each answer as the plain word number (e.g. 4318) and let
# sparse_categorical_crossentropy do the lookup during training. Identical
# maths, identical result, 0.69 MB instead of 8.66 GB.

print("inputs :", predictors.shape, f"({predictors.nbytes / 1e6:.1f} MB)")
print("answers:", label.shape, f"({label.nbytes / 1e6:.2f} MB)")

inputs : (171312, 15) (10.3 MB)
answers: (171312,) (0.69 MB)


In [38]:
model = Sequential([
    Embedding(total_words, 200),
    Bidirectional(LSTM(200)),
    Dense(total_words, activation='softmax')
])

# sparse_categorical_crossentropy == categorical_crossentropy, except it takes
# the answers as plain word numbers instead of one-hot rows.
model.compile(loss='sparse_categorical_crossentropy',
              optimizer='adam',
              metrics=['accuracy'])

model.build((None, max_length - 1))
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ (None, 15, 200)        │     2,526,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_2 (Bidirectional) │ (None, 400)            │       641,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 12633)          │     5,065,833 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 8,234,033 (31.41 MB)

 Trainable params: 8,234,033 (31.41 MB)

 Non-trainable params: 0 (0.00 B)

In [39]:
# batch_size=128 instead of the default 32: fewer, larger steps means far less
# time per pass on the GPU. 20 passes is plenty on this much text -- the old
# 100 passes mostly just memorised it.
history = model.fit(predictors, label, epochs=70, batch_size=128, verbose=1)

Epoch 1/70
1339/1339 ━━━━━━━━━━━━━━━━━━━━ 25s 17ms/step - accuracy: 0.0500 - loss: 6.7431
Epoch 2/70
1339/1339 ━━━━━━━━━━━━━━━━━━━━ 23s 17ms/step - accuracy: 0.0890 - loss: 6.0797
Epoch 3/70
1339/1339 ━━━━━━━━━━━━━━━━━━━━ 22s 17ms/step - accuracy: 0.1058 - loss: 5.7208
Epoch 4/70
1339/1339 ━━━━━━━━━━━━━━━━━━━━ 22s 17ms/step - accuracy: 0.1195 - loss: 5.4244
Epoch 5/70
1339/1339 ━━━━━━━━━━━━━━━━━━━━ 22s 17ms/step - accuracy: 0.1315 - loss: 5.1570
Epoch 6/70
1339/1339 ━━━━━━━━━━━━━━━━━━━━ 23s 17ms/step - accuracy: 0.1452 - loss: 4.9023
Epoch 7/70
1339/1339 ━━━━━━━━━━━━━━━━━━━━ 23s 17ms/step - accuracy: 0.1600 - loss: 4.6567
Epoch 8/70
1339/1339 ━━━━━━━━━━━━━━━━━━━━ 22s 17ms/step - accuracy: 0.1822 - loss: 4.4202
Epoch 9/70
1339/1339 ━━━━━━━━━━━━━━━━━━━━ 23s 17ms/step - accuracy: 0.2071 - loss: 4.1960
Epoch 10/70
1339/1339 ━━━━━━━━━━━━━━━━━━━━ 23s 17ms/step - accuracy: 0.2351 - loss: 3.9881
Epoch 11/70
1339/1339 ━━━━━━━━━━━━━━━━━━━━ 23s 17ms/step - accuracy: 0.2625 - loss: 3.7985
Epoch 12

In [40]:
model.save('word_lstm.keras')

In [41]:
def generate_story(seed_text, next_words, model, max_length, temperature=0.8):
    """Write next_words more words, starting from seed_text.

    temperature=0   -> always take the most likely word (tends to loop:
                       "and the king and the king...")
    temperature=0.8 -> pick from the likely words with some randomness
    """
    result = seed_text.lower()

    for _ in range(next_words):
        token_list = tokenizer.texts_to_sequences([result])[0][-(max_length - 1):]
        token_list = pad_sequences([token_list], maxlen=max_length - 1, padding='pre')

        probs = model.predict(token_list, verbose=0)[0]

        if temperature > 0:
            scaled = np.log(np.maximum(probs, 1e-9)) / temperature
            scaled = np.exp(scaled)
            probs = scaled / scaled.sum()
            predicted_index = int(np.random.choice(len(probs), p=probs))
        else:
            predicted_index = int(np.argmax(probs))

        # index_word is a direct lookup -- the old version looped over all
        # 12,633 words to find the match for every single word generated.
        result += " " + tokenizer.index_word.get(predicted_index, "")

    return result.strip()

In [42]:
# stories.txt is Shakespeare, so the seed has to sound like the dataset --
# every word in it must be a word the model has actually seen.
input_text = "first citizen before we proceed"

print(generate_story(input_text, 50, model, max_length, temperature=0.8))

first citizen before we proceed the officers o' the people in the refell'd of pates mistaking washed iniquity credit countrymen testimony highness' bond manners wolf argues gripe undo afflict cost mutiny saying ingratitude conclude prettiest mount bishop wed manners senator treading cheque wait rack nuptial mount bide helps purge nuptial bishop wed writing senator slipp'd


In [43]:
# A few more seeds from the same text to compare against.
for seed in ["what is the city but the people",
             "my lord i do beseech you",
             "o gentle romeo"]:
    print(seed.upper())
    print(generate_story(seed, 30, model, max_length, temperature=0.8))
    print()

WHAT IS THE CITY BUT THE PEOPLE
what is the city but the people with thee ever with to him i charge knell you puissant mowbray grey you as now you as you the york's wife ebb south weapon beggar's you as you as

MY LORD I DO BESEECH YOU
my lord i do beseech you hear me make it stand i ' good servants as the duke caused burn finds mistaking senator treading idly insinuate peer senator rascal hydra always wot woo'd you empire interr'd

O GENTLE ROMEO
o gentle romeo this was not even on a loud ere relent and reigns now eat bias destroy you as now now body the execute trumpet incaged now purge creatures stead car justify

